In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pymupdf
from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

from generation.llm_client import OllamaClient

In [2]:
path = Path("..") / "data" / "manuals" / "fronius_symo_10-20kw.pdf"
doc = pymupdf.open(path)
pages = []

for page in doc:
    text = page.get_text()
    pages.append({"page": page.number + 1, "text": text})

print(len(pages))

46


In [3]:
embedding_model = TextEmbedding("BAAI/bge-base-en-v1.5")
print("The model BAAI/bge-base-en-v1.5 is ready to use.")

texts = [p["text"] for p in pages]
embeddings_list = list(embedding_model.embed(texts))

print(f"Embedded {len(embeddings_list)} pages, dimension: {len(embeddings_list[0])}")

The model BAAI/bge-base-en-v1.5 is ready to use.
Embedded 46 pages, dimension: 768


In [4]:
client = QdrantClient(":memory:")

client.create_collection(
    collection_name="manuals",
    vectors_config=VectorParams(size=len(embeddings_list[0]), distance=Distance.COSINE),
)

points = [
    PointStruct(id=i, vector=embeddings_list[i].tolist(), payload=pages[i])
    for i in range(len(pages))
]

client.upsert("manuals", points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [5]:
question = "What does error 503 mean and how do I fix it?"
embedding_question = list(embedding_model.embed([question]))[0].tolist()

result = client.query_points(collection_name="manuals", query=embedding_question, limit=3).points

for r in result:
    print(f"Page {r.payload['page']}")
    print(r.payload["text"][:300])
    print("...")

Page 3
3
EN
Contents
Installation location and position ................................................................................................................
5
Explanation of safety symbols ..........................................................................................................
...
Page 5
5
EN
Installation location and position
Explanation of 
safety symbols
If you see any of the symbols depicted in the "Safety rules" chapter, special care is re-
quired.
Reliability
DANGER! Indicates immediate and real danger. If it is not avoided, death or se-
rious injury will result.
WARNING! Indi
...
Page 26
26
IMPORTANT! Observe the ESD guidelines when handling option cards. 
IMPORTANT! Only one Fronius Datamanager in master mode is permitted per Fronius 
Solar Net ring. Switch any other Fronius Datamanagers to slave mode or remove them.
Seal off the unoccupied option card slot by replacing the cover (
...


In [ ]:
context = "\n\n".join([f"[Page {r.payload['page']}]\n{r.payload['text']}" for r in result])

prompt = f"""You are a solar inverter technician assistant.
Answer based only on the manual excerpts below.
Cite the page number in your answer.
{context}
Question: {question}
Answer:"""

llm = OllamaClient()
answer = llm.generate(prompt)
print(answer)

Error 503 typically indicates that a web server is either down or misconfigured. This error is commonly seen when a website is not responding or is not accessible. To fix this issue, you should:

1. **Check the Website**: Ensure that the website URL is correct and that the website is operational.

2. **Wait or Refresh**: Sometimes, a simple refresh of the browser or a brief wait can resolve the issue.

3. **Contact Support**: If the issue persists, contact the website's support team. They can provide specific assistance based on their knowledge of the server configuration.

4. **Contact Network Administrator**: If you are accessing the website from an organization's network, contact your network administrator to check if there are any network issues.

5. **DNS Issues**: Check if there are any DNS resolution problems. This can be addressed by clearing your DNS cache or contacting your network administrator for DNS configuration issues.

6. **Firewall or Proxy Settings**: Ensure that you

In [7]:
question_2 = "How do I connect the AC cables to the inverter?"
embedding_question = list(embedding_model.embed([question_2]))[0].tolist()

result = client.query_points(collection_name="manuals", query=embedding_question, limit=3).points

for r in result:
    print(f"Page {r.payload['page']}")
    print(r.payload["text"][:300])
    print("...")

Page 13
13
EN
Connecting the inverter to the public grid (AC)
Safety
Monitoring the 
Grid
Type of AC cable
The following types of AC cable can be connected to the AC terminals of the inverter:
Connecting alu-
minum cables
The AC-side terminals are designed for connecting single-wire, round, aluminum cables.
...
Page 15
15
EN
Connecting the 
inverter to the 
public grid (AC)
IMPORTANT! The PE ground conductor of the AC cable must be laid in such a way that it 
is the last to be disconnected in the event that the strain-relief device should fail.
This can be ensured, for example, by making it somewhat longer and by 
...
Page 28
28
Clipping the Inverter onto the Wall Bracket
Attaching the in-
verter to the wall 
bracket
Two people are required to attach the inverter to the wall bracket, as it is extremely heavy. 
The fastening screws in the data communication area of the inverter are used for securing 
the inverter to the w
...


In [ ]:
context = "\n\n".join([f"[Page {r.payload['page']}]\n{r.payload['text']}" for r in result])

prompt = f"""You are a solar inverter technician assistant.
Answer based only on the manual excerpts below.
Cite the page number in your answer.

{context}

Question: {question_2}
Answer:"""

llm = OllamaClient()
answer = llm.generate(prompt)
print(answer)

To connect AC cables to the inverter, follow these steps based on the manual excerpts:

1. **Prepare the AC Cables**: 
   - The AC-side terminals of the inverter are designed to connect single-wire, round, aluminum cables.
   - Ensure that the cables are clean and free from the non-conducting oxide layer.

2. **Clean and Prepare the Cable Ends**:
   - Carefully remove the oxide layer from the stripped ends of the cable, e.g., using a knife.
   - Do not use brushes, files, or sandpaper as they may trap aluminum particles and transfer them to other cables.

3. **Apply Neutral Grease**:
   - After cleaning the cable ends, apply a neutral grease, such as an acid- and alkali-free Vaseline.
   - The type of grease depends on the material of the cable, i.e., either copper or aluminum.
   - The application must be done immediately after cleaning.

4. **Connect the Cable to the Terminal**:
   - Connect the prepared cable to the inverter terminal immediately after applying the grease.

5. **Reco

## What broke / observations

  - **Error 503 not found**: The Fronius Symo manual does not appear to contain error code 503.
    The model ignored the context and answered from general knowledge - hallucination.
  - **Page citation inconsistent**: The model does not reliably cite the page number even when instructed to.
    Needs prompt engineering or structured output in a future sprint.
  - **Text quality**: PyMuPDF extracts raw text without layout awareness - tables and figures are lost.
    Docling (story 3.3) will replace this step.
  - **Next steps shaped by this**: reranker, stricter prompting, Docling parsing, real error-code test set.
